In [ ]:
# imports
from pathlib import Path

import torch
from torch import nn

from ray import tune
from ray.tune import Tuner, TuneConfig
from ray.tune.schedulers import ASHAScheduler

from sklearn.metrics import (
    accuracy_score,
    recall_score,
 )

from src.data_utils import (
    FullDataStreamer,
    split_train_validation,
    calculate_class_weights,
 )
from src import datasets
from src.networks import ParallelCNN2D

In [ ]:
# configuratie en bestanden inladen
import tomllib

datadir = Path("../data")
configfile = Path("config.toml")

with configfile.open("rb") as f:
    config = tomllib.load(f)

trainfile = datadir / (
    config["arrhythmia"] + "_train.parq"
)

testfile = datadir / (
    config["arrhythmia"] + "_test.parq"
)

trainfile, testfile

In [ ]:
# Dataset maken. Alleen de trainingdata wordt opgesplitst voor modelselectie.
shape = (16, 12)
full_traindataset = datasets.HeartDataset2D(trainfile, target="target", shape=shape)
testdataset = datasets.HeartDataset2D(testfile, target="target", shape=shape)
traindataset, validdataset = split_train_validation(
    full_traindataset, validation_fraction=0.2, seed=42
)
len(traindataset), len(validdataset), len(testdataset)


In [ ]:
# Class weights uitsluitend op trainingdata berekenen
class_weights = calculate_class_weights(traindataset)
classes = ["N", "S", "V", "F", "Q"]
for cls, weight in zip(classes, class_weights):
    print(f"{cls}: weight={weight:.3f}")


In [ ]:
# Testen of model nog werkt na parameters toevoegen

model = ParallelCNN2D(
    num_classes=5,
    filters=16,
    matrixshape=(16, 12),
    large_kernel=5,
    dropout=0.2,
)

teststreamer = FullDataStreamer(
    testdataset,
    batch_size=32,
    shuffle=False,
)

x, y = next(teststreamer.stream())

with torch.no_grad():
    output = model(x)

print(x.shape)
print(output.shape)

In [ ]:
# Ray train functie

def train_tune_model(config):
    trainstreamer = FullDataStreamer(
        traindataset,
        batch_size=config["batch_size"],
    )

    validstreamer = FullDataStreamer(
        validdataset,
        batch_size=config["batch_size"],
        shuffle=False,
    )

    model = ParallelCNN2D(
        num_classes=5,
        filters=config["filters"],
        matrixshape=(16, 12),
        large_kernel=config["large_kernel"],
        dropout=config["dropout"],
    )

    loss_fn = nn.CrossEntropyLoss(
        weight=class_weights,
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config["learning_rate"],
        weight_decay=config["weight_decay"],
    )

    for epoch in range(config["epochs"]):
        model.train()

        train_data = trainstreamer.stream()

        for _ in range(len(trainstreamer)):
            X, y = next(train_data)

            optimizer.zero_grad()

            logits = model(X)

            loss = loss_fn(
                logits,
                y,
            )

            loss.backward()
            optimizer.step()

        model.eval()

        y_true = []
        y_pred = []

        valid_data = validstreamer.stream()

        with torch.no_grad():
            for _ in range(len(validstreamer)):
                X, y = next(valid_data)

                predictions = (
                    model(X)
                    .argmax(dim=1)
                )

                y_true.extend(
                    y.numpy()
                )

                y_pred.extend(
                    predictions.numpy()
                )

        recalls = recall_score(
            y_true,
            y_pred,
            labels=[0, 1, 2, 3, 4],
            average=None,
            zero_division=0,
        )

        macro_recall = recalls.mean()

        abnormal_recall = recalls[
            [1, 2, 3, 4]
        ].mean()

        accuracy = accuracy_score(
            y_true,
            y_pred,
        )

        tune.report(
            {
                "epoch": epoch + 1,
                "loss": float(loss.item()),
                "accuracy": accuracy,
                "macro_recall": macro_recall,
                "abnormal_recall": abnormal_recall,
                "recall_N": recalls[0],
                "recall_S": recalls[1],
                "recall_V": recalls[2],
                "recall_F": recalls[3],
                "recall_Q": recalls[4],
            }
        )

In [ ]:
# Zoekruimte

search_space = {
    "learning_rate": tune.loguniform(
        1e-4,
        1e-2,
    ),
    "filters": tune.choice(
        [8, 16, 32, 64]
    ),
    "batch_size": tune.choice(
        [16, 32, 64]
    ),
    "weight_decay": tune.loguniform(
        1e-6,
        1e-3,
    ),
    "dropout": tune.uniform(
        0.0,
        0.4,
    ),
    "large_kernel": tune.choice(
        [5, 7]
    ),
    "epochs": 10,
}

# filters: modelcapaciteit
# learning_rate: grootte van gewichtsupdates
# batch_size: stabiliteit en frequentie van updates
# weight_decay: regularisatie
# dropout: regularisatie
# large_kernel: schaal van de tweede route

In [ ]:
# Schedular

scheduler = ASHAScheduler(
    time_attr="training_iteration",
    metric="macro_recall",
    mode="max",
    max_t=10,
    grace_period=3,
    reduction_factor=2,
)

In [ ]:
# Tunen

trainable = tune.with_resources(
    train_tune_model,
    resources={
        "cpu": 1,
    },
)

In [ ]:
tuner = Tuner(
    trainable,
    param_space=search_space,
    tune_config=TuneConfig(
        scheduler=scheduler,
        num_samples=20,
        max_concurrent_trials=1,
    ),
    run_config=tune.RunConfig(
        name="parallel_2d_hypertuning",
        storage_path=str(
            Path("ray_results").resolve()
        ),
    ),
)

In [ ]:
results = tuner.fit()

In [ ]:
# Beste configuratie

best_result = results.get_best_result(
    metric="macro_recall",
    mode="max",
)

print("Beste configuratie:")
print(best_result.config)

print("\nBeste macro-recall:")
print(
    best_result.metrics["macro_recall"]
)

print("\nRecall per klasse:")

for cls in classes:
    print(
        f"{cls}: "
        f"{best_result.metrics[f'recall_{cls}']:.3f}"
    )


In [ ]:
# Alles in dataframe zetten

tuning_results = results.get_dataframe()

tuning_results.head()


In [ ]:
# Pak relevante kolommen

tuning_summary = tuning_results[
    [
        "config/learning_rate",
        "config/filters",
        "config/batch_size",
        "config/weight_decay",
        "config/dropout",
        "config/large_kernel",
        "accuracy",
        "macro_recall",
        "abnormal_recall",
        "recall_N",
        "recall_S",
        "recall_V",
        "recall_F",
        "recall_Q",
    ]
].copy()

tuning_summary = tuning_summary.sort_values(
    "macro_recall",
    ascending=False,
)

tuning_summary.head(10).round(4)

In [ ]:
# Opslaan

Path("results").mkdir(
    parents=True,
    exist_ok=True,
)

tuning_summary.to_csv(
    "results/ray_tuning_results.csv",
    index=False,
)

In [ ]:
# BEste configuratie opnieuw trainen

import pandas as pd

tuning_results = pd.read_csv(
    "results/ray_tuning_results.csv"
)

best_config = tuning_results.loc[
    tuning_results["macro_recall"].idxmax()
]

best_config

In [ ]:
from src.helpers import run_experiment

best_model_results = run_experiment(
    model_function=lambda: ParallelCNN2D(
        num_classes=5,
        filters=int(best_config["config/filters"]),
        matrixshape=(16, 12),
        large_kernel=int(best_config["config/large_kernel"]),
        dropout=float(best_config["config/dropout"]),
    ),
    experiment_name="Parallelle 2D CNN hypertuned",
    trainstreamer=FullDataStreamer(
        traindataset,
        batch_size=int(best_config["config/batch_size"]),
    ),
    validstreamer=FullDataStreamer(
        validdataset,
        batch_size=int(best_config["config/batch_size"]),
        shuffle=False,
    ),
    teststreamer=FullDataStreamer(
        testdataset,
        batch_size=int(best_config["config/batch_size"]),
        shuffle=False,
    ),
    loss_fn=nn.CrossEntropyLoss(
        weight=class_weights,
    ),
    class_weights=True,
    parallel=True,
    runs=5,
    epochs=15,
    learning_rate=float(best_config["config/learning_rate"]),
    weight_decay=float(best_config["config/weight_decay"]),
    filename="results/results.csv",
)

In [ ]:
tuning_results[
    [
        "macro_recall",
        "config/learning_rate",
        "config/batch_size",
        "config/filters",
        "config/dropout",
        "config/weight_decay",
        "config/large_kernel",
    ]
].sort_values(
    "macro_recall",
    ascending=False,
).head(10)